In [1]:
import pandas as pd
import plotly.express as px

In [2]:
partidos = pd.read_csv("../data/raw/2016-2024_liga_mx.csv")
players = pd.read_csv("../data/raw/players.csv")
valuations = pd.read_csv("../data/raw/player_valuations.csv")

partidos.info()
#partidos.head()

<class 'pandas.DataFrame'>
RangeIndex: 2876 entries, 0 to 2875
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id                     2876 non-null   int64  
 1   referee                2605 non-null   str    
 2   timezone               2876 non-null   str    
 3   date                   2876 non-null   str    
 4   venue_id               1976 non-null   float64
 5   venue_name             2876 non-null   str    
 6   venue_city             2695 non-null   str    
 7   season                 2876 non-null   int64  
 8   round                  2876 non-null   str    
 9   home_team              2876 non-null   str    
 10  away_team              2876 non-null   str    
 11  home_win               2066 non-null   object 
 12  away_win               2066 non-null   object 
 13  home_goals             2813 non-null   float64
 14  away_goals             2813 non-null   float64
 15  home_goals_half

In [3]:
#limpiar datos
partidos["date"] = pd.to_datetime(partidos["date"],utc = True)
partidos["year"] = partidos["date"].dt.year

#filas sin goles
partidos = partidos.dropna(subset=["home_goals","away_goals"])
#partidos.head()

def resultado(row):
    if row["home_goals"] > row["away_goals"]:
        return "gana local"
    elif row["home_goals"] < row["away_goals"]:
        return "gana visitante"
    else:
        return "empate"
    
partidos["resultado"] = partidos.apply(resultado, axis=1)

columnas_partidos = ["date", "year", "home_team", "away_team", "home_goals", "away_goals", "resultado"]
#columnas_partidos = partidos[["date", "home_team", "away_team", "home_goals", "away_goals", "resultado"]].head()

#partidos.info()

In [10]:
#solo jugadores de la liga-mx
#valuations.info()
#valuations["player_club_domestic_competition_id"].unique()
val_mx = valuations[valuations["player_club_domestic_competition_id"] =="MEX1"].copy()
val_mx["date"] = pd.to_datetime(val_mx["date"])
val_mx["year"] = val_mx["date"].dt.year
#val_mx.head()
ultima_valuacion = val_mx.sort_values("date").groupby("player_id").last().reset_index()
ultima_valuacion = ultima_valuacion.merge(players[["player_id","name", "position"]],
                                          on="player_id", how="left")
ultima_valuacion.head()

#convertir a millones de euros
ultima_valuacion["market_value_millon"] = ultima_valuacion["market_value_in_eur"] / 1_000_000

#top 15 jugadores mas valiosos
top_jugadores = ultima_valuacion.sort_values("market_value_millon", ascending=False).head(15)
#top_jugadores

"""fig = px.bar(top_jugadores,
             x = "market_value_millon",
             y = "name",
             orientation= "h",
             text_auto=".2f",
             labels={"market_value_millon": "Valor de Mercado (Millones €)",
                     "name": "Jugador"},
             title = "Top 15 jugadores más valiosos de la Liga MX",
             color = "market_value_millon",
             color_continuous_scale="Viridis",
             template = "plotly_dark")

fig.update_layout(yaxis={"categoryorder":"total ascending"})
fig.show()"""
top_jugadores

,player_id,date,market_value_in_eur,current_club_name,current_club_id,player_club_domestic_competition_id,year,name,position,market_value_millon
575,837619,2026-05-20,15000000,Deportivo Guadalajara,6711,MEX1,2026,Armando González,Attack,15.0
451,602324,2026-05-20,12000000,CD Cruz Azul,3711,MEX1,2026,Érik Lira,Midfield,12.0
99,184892,2022-10-31,10000000,Tigres UANL,7055,MEX1,2022,Florian Thauvin,Attack,10.0
249,351505,2026-05-20,10000000,CF América,3631,MEX1,2026,Álex Zendejas,Attack,10.0
217,316889,2017-06-23,10000000,CF Pachuca,4035,MEX1,2017,Hirving Lozano,Attack,10.0
242,342385,2020-01-10,10000000,CF América,3631,MEX1,2020,Guido Rodríguez,Midfield,10.0
700,1155723,2026-05-20,10000000,Club Tijuana,13353,MEX1,2026,Gilberto Mora,Midfield,10.0
170,272642,2025-12-29,10000000,CF América,3631,MEX1,2025,Allan Saint-Maximin,Attack,10.0
442,597728,2026-05-20,10000000,Deportivo Toluca,1804,MEX1,2026,Marcel Ruiz,Midfield,10.0
318,416090,2024-06-14,10000000,CF América,3631,MEX1,2024,Julián Quiñones,Attack,10.0


In [5]:
jugador_A = "André-Pierre Gignac"
jugador_B = "Rogelio Funes Mori"

radar = ultima_valuacion[ultima_valuacion["name"].isin([jugador_A,jugador_B])]

historico = val_mx.merge(players[["player_id","name"]],on="player_id", how="left")
historico["market_value_millon"] = historico["market_value_in_eur"] / 1_000_000
columnas_historico = ["player_id", "name", "date", "year", "market_value_millon"]
historico_procesado = historico[columnas_historico]
filtrado = historico_procesado[historico_procesado["name"].isin([jugador_A, jugador_B])].copy()

fig_radar = px.line(filtrado,
                    x="date",
                    y = "market_value_millon",
                    color= "name",
                    markers=True,
                    labels ={"date": "Fecha de Valuacion",
                             "market_value_millon": "Valor de Mercado (Millones €)",
                             "name": "Futbolista"},
                    title = f"Historial financiero: {jugador_A} vs {jugador_B}",
                    template="plotly_dark",
                    color_discrete_sequence=["#00FFCC", "#FF3366"])
fig_radar.update_layout(hovermode = "x unified")
fig_radar.show()

In [6]:

local = partidos[["year", "home_team", "resultado"]].copy()
local.columns = ["year", "Equipo", "res_original"]
local["Rendimiento"] = local["res_original"].map({"gana local": "Victoria", "gana visitante": "Derrota", "empate": "Empate"})

visit = partidos[["year", "away_team", "resultado"]].copy()
visit.columns = ["year", "Equipo", "res_original"]
visit["Rendimiento"] = visit["res_original"].map({"gana visitante": "Victoria", "gana local": "Derrota", "empate": "Empate"})


historico_rendimiento = pd.concat([local, visit])
rendimiento_procesado = historico_rendimiento.groupby(["year", "Equipo", "Rendimiento"]).size().reset_index(name="Partidos")
rendimiento_procesado

#rendimiento_procesado.to_parquet("../data/processed/rendimiento_equipos.parquet")


,year,Equipo,Rendimiento,Partidos
0,2016,Atlas,Derrota,6
1,2016,Atlas,Empate,7
2,2016,Atlas,Victoria,4
3,2016,Club America,Derrota,3
4,2016,Club America,Empate,11
...,...,...,...,...
495,2024,Toluca,Empate,11
496,2024,Toluca,Victoria,19
497,2024,U.N.A.M. - Pumas,Derrota,11
498,2024,U.N.A.M. - Pumas,Empate,12


In [7]:
#partidos[columnas_partidos].to_parquet("../data/processed/partidos.parquet",index= False)
#ultima_valuacion.to_parquet("../data/processed/ultimas_valuaciones.parquet", index= False)
#historico_procesado.to_parquet("../data/processed/historico_valuaciones.parquet", index=False)

